# Transformações e Escalonamento - Parte 1
## Conceitos de Normalização e Padronização

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 07 - Parte 1: Conceitos Fundamentais

---

# ==================================================
# 1. INTRODUÇÃO À NORMALIZAÇÃO E PADRONIZAÇÃO
# ==================================================

OBJETIVOS DESTA PARTE:
- Compreender a importância do escalonamento de features em ciência de dados
- Aprender as técnicas fundamentais: Min-Max Scaling e Standardization
- Entender Robust Scaling para dados com outliers
- Implementar e comparar diferentes técnicas de escalonamento
- Saber quando usar cada abordagem

In [ ]:
# @title
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.datasets import fetch_california_housing

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ==================================================
# 2. POR QUE ESCALONAR FEATURES?
# ==================================================

O PROBLEMA DE DIFERENTES ESCALAS:

1. **Comparação injusta entre variáveis**  
   - Variáveis em escalas maiores dominam o modelo  
   - Exemplo: comparar idade (0-100) vs. renda (0-1.000.000)  

2. **Impacto em algoritmos baseados em distância**  
   - k-Nearest Neighbors (KNN)  
   - K-means clustering  
   - Support Vector Machines (SVM)  
   - Principal Component Analysis (PCA)  

3. **Problemas com métodos baseados em gradiente**  
   - Convergência mais lenta  
   - Instabilidade numérica  
   - Dificulta a escolha da taxa de aprendizado  

4. **Exemplos concretos onde o escalonamento é crucial**  
   - Regularização em modelos lineares (L1/L2)  
   - Redes neurais  
   - Análise de componentes principais (PCA)  
   - Algoritmos de clusterização

# ==================================================
# 3. DIFERENÇA ENTRE NORMALIZAÇÃO E PADRONIZAÇÃO
# ==================================================

NORMALIZAÇÃO:
- Reescala os valores para um intervalo específico, geralmente [0,1] ou [-1,1]
- Preserva a distribuição original, apenas "comprime" para o novo intervalo
- Fórmula Min-Max: X_norm = (X - X_min) / (X_max - X_min)
- Sensível a outliers (pois usa min e max)

PADRONIZAÇÃO (Z-SCORE):
- Reescala dados para ter média 0 e desvio padrão 1
- Centra os dados em relação à média e escala pelo desvio padrão
- Fórmula: X_std = (X - μ) / σ
- Menos sensível a outliers que a normalização (usa média e desvio padrão)

PRINCIPAIS DIFERENÇAS:
- Intervalo: Normalização resulta em intervalo fixo; padronização não
- Outliers: Padronização geralmente lida melhor com outliers
- Distribuição: Padronização preserva melhor a forma da distribuição
- Aplicação: Normalização é preferível quando o algoritmo requer certos limites

# ==================================================
# 4. CRIANDO UM CONJUNTO DE DADOS PARA EXEMPLOS
# ==================================================

### California Housing dataset

Neste dataset, temos informações sobre diferentes **distritos**.  
Para cada distrito, são fornecidos:

- **Dados demográficos**: renda média da população, número de habitantes e como as casas estão ocupadas.  
- **Localização geográfica**: latitude e longitude.  
- **Características das casas**: quantidade média de cômodos, número médio de quartos e a idade média das construções.  

⚠️ Essas informações **não descrevem uma casa específica**, mas sim **valores médios ou medianos do distrito como um todo**.


In [ ]:
# @title
# Carregando o California Housing dataset
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

# Exibindo as primeiras linhas e informações
print("Primeiras linhas do dataset:")
print(X.head())

# Exibindo estatísticas descritivas
print("\nEstatísticas descritivas:")
print(X.describe())

In [ ]:
# @title
# Adicionando a variável target ao DataFrame para facilitar a análise
df = X.copy()
df['target'] = y

# Visualizando a distribuição das variáveis para entender suas escalas
plt.figure(figsize=(15, 10))
for i, column in enumerate(X.columns):
    plt.subplot(3, 3, i+1)
    sns.histplot(X[column], kde=True)
    plt.title(f'Distribuição de {column}')
plt.tight_layout()
plt.show()

In [ ]:
# @title
# Visualizando a escala das variáveis com boxplots
plt.figure(figsize=(12, 6))
sns.boxplot(data=X)
plt.title('Comparação de Escalas entre Variáveis')
plt.xticks(rotation=45)
plt.ylabel('Valor')
plt.show()

In [ ]:
# @title
# Calculando a matriz de correlação
correlation_matrix = df.corr()

# Visualizando a matriz de correlação
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlação das Variáveis')
plt.show()

# ==================================================
# 5. IMPLEMENTANDO MIN-MAX SCALING (NORMALIZAÇÃO)
# ==================================================

**Min-Max Scaling (Normalização)**

Esta técnica reescala os dados para um intervalo específico, geralmente [0, 1]. A transformação é dada por:

$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$

Onde:
- $X$ é o valor original
- $X_{min}$ é o valor mínimo de $X$
- $X_{max}$ é o valor máximo de $X$
- $X_{norm}$ é o valor normalizado

Para um intervalo diferente [a, b], a fórmula é:

$X_{norm} = a + \frac{(X - X_{min})(b - a)}{X_{max} - X_{min}}$

**Quando usar Min-Max Scaling:**
- Quando precisamos de valores em um intervalo específico
- Quando a distribuição não é Normal (Gaussiana) ou quando não sabemos a distribuição
- Para algoritmos que requerem valores positivos (ex: alguns algoritmos de NLP)

In [ ]:
# @title
# Implementando Min-Max Scaling
scaler = MinMaxScaler()
X_minmax = scaler.fit_transform(X)

# Convertendo para DataFrame para facilitar a visualização
X_minmax_df = pd.DataFrame(X_minmax, columns=X.columns)

# Exibindo estatísticas após normalização
print("Estatísticas após Min-Max Scaling:")
print(X_minmax_df.describe())

⚠️ **Observação importante**:  
No exemplo aplicamos o *Min-Max Scaling* em todas as variáveis do conjunto de dados para fins de demonstração.  
Porém, em aplicações reais **não é comum escalar todas as variáveis**.  

- Variáveis como **latitude** e **longitude** não devem ser escaladas com Min-Max, pois são coordenadas geográficas e perderiam seu significado.  
- O *scaling* deve ser aplicado principalmente em variáveis numéricas contínuas (como renda, número de quartos, idade da casa etc.), quando isso ajuda o algoritmo de aprendizado de máquina a convergir melhor.


In [ ]:
# @title Visualizando algumas variáveis antes e depois da normalização
feature_to_plot = 'MedInc'  # Podemos mudar para ver outras variáveis

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(X[feature_to_plot], kde=True)
plt.title(f'{feature_to_plot} - Original')

plt.subplot(1, 2, 2)
sns.histplot(X_minmax_df[feature_to_plot], kde=True)
plt.title(f'{feature_to_plot} - Normalizado')

plt.tight_layout()
plt.show()

In [ ]:
# @title Comparando a escala das variáveis antes e depois com boxplots
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.boxplot(data=X)
plt.title('Variáveis Originais')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.boxplot(data=X_minmax_df)
plt.title('Variáveis após Min-Max Scaling')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# ==================================================
# 6. IMPLEMENTANDO STANDARDIZATION (PADRONIZAÇÃO)
# ==================================================

**Standardization (Padronização / Z-score)**

Esta técnica transforma os dados para que tenham média 0 e desvio padrão 1. A transformação é dada por:

$X_{std} = \frac{X - \mu}{\sigma}$

Onde:
- $X$ é o valor original
- $\mu$ é a média da amostra
- $\sigma$ é o desvio padrão da amostra
- $X_{std}$ é o valor padronizado

**Quando usar Standardization:**
- Quando os dados seguem aproximadamente uma distribuição normal
- Para algoritmos que assumem que os dados estão normalmente distribuídos
- Para algoritmos baseados em distância euclidiana (linha reta que liga dois pontos)
- Quando os valores extremos (outliers) não são tão significativos

In [ ]:
# @title
# Implementando Standardization (Z-score)
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# Convertendo para DataFrame para facilitar a visualização
X_std_df = pd.DataFrame(X_std, columns=X.columns)

# Exibindo estatísticas após padronização
print("Estatísticas após Standardization:")
print(X_std_df.describe())

ℹ️ **Observação**:  
Após a padronização, a média deve ser 0 e o desvio padrão 1.  
Na prática, os valores podem aparecer como `-1.101617e-17` para a média ou `1.000024` para o desvio padrão.  
Isso acontece por conta do **arredondamento interno da CPU** e da forma como o computador representa números em ponto flutuante.  
Ou seja, não é um erro — apenas uma limitação da precisão numérica.


In [ ]:
# @title
# Verificando se a média está próxima de 0 e o desvio padrão próximo de 1
print("Média das variáveis padronizadas:")
print(X_std_df.mean())

print("\nDesvio padrão das variáveis padronizadas:")
print(X_std_df.std())

In [ ]:
# @title
# Visualizando algumas variáveis antes e depois da padronização
feature_to_plot = 'MedInc'  # Podemos mudar para ver outras variáveis

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(X[feature_to_plot], kde=True)
plt.title(f'{feature_to_plot} - Original')

plt.subplot(1, 2, 2)
sns.histplot(X_std_df[feature_to_plot], kde=True)
plt.title(f'{feature_to_plot} - Padronizado')

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Comparando a escala das variáveis após padronização com boxplots
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.boxplot(data=X)
plt.title('Variáveis Originais')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.boxplot(data=X_std_df)
plt.title('Variáveis após Standardization')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# ==================================================
# 7. COMPARAÇÃO: MIN-MAX SCALING VS. STANDARDIZATION
# ==================================================

In [ ]:
# @title
# Visualizando a diferença entre Min-Max Scaling e Standardization
plt.figure(figsize=(15, 5))

# Selecionando duas variáveis para comparação
feature1 = 'MedInc'
feature2 = 'Population'

# Dados originais
plt.subplot(1, 3, 1)
plt.scatter(X[feature1], X[feature2], alpha=0.3)
plt.title('Dados Originais')
plt.xlabel(feature1)
plt.ylabel(feature2)

# Após Min-Max Scaling
plt.subplot(1, 3, 2)
plt.scatter(X_minmax_df[feature1], X_minmax_df[feature2], alpha=0.3)
plt.title('Após Min-Max Scaling')
plt.xlabel(feature1)
plt.ylabel(feature2)

# Após Standardization
plt.subplot(1, 3, 3)
plt.scatter(X_std_df[feature1], X_std_df[feature2], alpha=0.3)
plt.title('Após Standardization')
plt.xlabel(feature1)
plt.ylabel(feature2)

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Visualizando a distribuição antes e depois de cada transformação
# Selecionando uma variável com distribuição assimétrica e outra mais próxima da normal
skewed_feature = 'AveOccup'  # Tipicamente assimétrica
normal_feature = 'HouseAge'  # Mais próxima da normal

plt.figure(figsize=(15, 10))

# Variável assimétrica
plt.subplot(2, 3, 1)
sns.histplot(X[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Original')

plt.subplot(2, 3, 2)
sns.histplot(X_minmax_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Min-Max')

plt.subplot(2, 3, 3)
sns.histplot(X_std_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Standardized')

# Variável mais normal
plt.subplot(2, 3, 4)
sns.histplot(X[normal_feature], kde=True)
plt.title(f'{normal_feature} - Original')

plt.subplot(2, 3, 5)
sns.histplot(X_minmax_df[normal_feature], kde=True)
plt.title(f'{normal_feature} - Min-Max')

plt.subplot(2, 3, 6)
sns.histplot(X_std_df[normal_feature], kde=True)
plt.title(f'{normal_feature} - Standardized')

plt.tight_layout()
plt.show()

**Principais diferenças observadas:**

1. **Intervalo de valores:**
   - Min-Max Scaling: Valores entre 0 e 1
   - Standardization: Valores geralmente entre -3 e 3 (para dados aproximadamente normais), mas sem limites específicos

2. **Efeito na distribuição:**
   - Min-Max Scaling: Preserva a forma da distribuição, apenas a comprime para o intervalo [0,1]
   - Standardization: Mantém a forma, mas centra em 0 e escala para ter desvio padrão 1

3. **Outliers:**
   - Min-Max Scaling: Mais sensível a outliers, pois usa min e max
   - Standardization: Menos sensível, pois usa média e desvio padrão

4. **Interpretabilidade:**
   - Min-Max Scaling: Fácil interpretar como porcentagem ou proporção
   - Standardization: Interpretação em termos de desvios padrão da média

# ==================================================
# 8. ROBUST SCALING
# ==================================================

**Robust Scaling**

Esta técnica usa estatísticas robustas (mediana e quartis) em vez de média e desvio padrão, tornando-a menos sensível a outliers. A transformação é dada por:

$X_{robust} = \frac{X - X_{mediana}}{IQR}$

Onde:
- $X$ é o valor original
- $X_{mediana}$ é a mediana da amostra
- $IQR$ é o intervalo interquartil (Q3 - Q1)
- $X_{robust}$ é o valor após escalonamento robusto

**Quando usar Robust Scaling:**
- Quando os dados contêm muitos outliers
- Quando queremos preservar informações sobre outliers (sem deixá-los dominar)
- Para algoritmos sensíveis a outliers, como regressão linear

In [ ]:
# @title
# Implementando Robust Scaling
robust_scaler = RobustScaler()
X_robust = robust_scaler.fit_transform(X)

# Convertendo para DataFrame
X_robust_df = pd.DataFrame(X_robust, columns=X.columns)

# Exibindo estatísticas após escalonamento robusto
print("Estatísticas após Robust Scaling:")
print(X_robust_df.describe())

In [ ]:
# @title
# Verificando a mediana (deve estar próxima de 0) e IQR (deve estar próximo de 1)
print("Mediana das variáveis após Robust Scaling:")
print(X_robust_df.median())

print("\nIQR das variáveis após Robust Scaling:")
for column in X_robust_df.columns:
    Q1 = X_robust_df[column].quantile(0.25)
    Q3 = X_robust_df[column].quantile(0.75)
    IQR = Q3 - Q1
    print(f"{column}: {IQR:.4f}")

In [ ]:
# @title
# Comparando Original, Min-Max, Standard e Robust para uma variável com outliers
# Vamos usar 'AveOccup' que tipicamente tem alguns outliers
feature_with_outliers = 'AveOccup'

plt.figure(figsize=(15, 12))

# Original
plt.subplot(2, 2, 1)
sns.boxplot(x=X[feature_with_outliers])
plt.title(f'{feature_with_outliers} - Original')

# Min-Max
plt.subplot(2, 2, 2)
sns.boxplot(x=X_minmax_df[feature_with_outliers])
plt.title(f'{feature_with_outliers} - Min-Max')

# Standard
plt.subplot(2, 2, 3)
sns.boxplot(x=X_std_df[feature_with_outliers])
plt.title(f'{feature_with_outliers} - Standard')

# Robust
plt.subplot(2, 2, 4)
sns.boxplot(x=X_robust_df[feature_with_outliers])
plt.title(f'{feature_with_outliers} - Robust')

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Comparando histogramas
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(X_minmax_df[feature_with_outliers], kde=True)
plt.title(f'{feature_with_outliers} - Min-Max')
plt.xlim(0, 1)  # Limitando para ver melhor a distribuição principal

plt.subplot(1, 3, 2)
sns.histplot(X_std_df[feature_with_outliers], kde=True)
plt.title(f'{feature_with_outliers} - Standard')
plt.xlim(-3, 3)  # Limitando para ver melhor a distribuição principal

plt.subplot(1, 3, 3)
sns.histplot(X_robust_df[feature_with_outliers], kde=True)
plt.title(f'{feature_with_outliers} - Robust')
plt.xlim(-3, 3)  # Limitando para ver melhor a distribuição principal

plt.tight_layout()
plt.show()

**Observações sobre Robust Scaling:**

1. **Tratamento de outliers:**
   - Robust Scaling mostra menos distorção devido a outliers
   - Outliers continuam sendo outliers, mas têm menos influência
   - A escala é determinada pelo IQR, que ignora valores extremos

2. **Comparação com Standard Scaling:**
   - Robust: usa mediana e IQR (resistentes a outliers)
   - Standard: usa média e desvio padrão (sensíveis a outliers)

3. **Quando preferir Robust Scaling:**
   - Dados com muitos outliers ou valores extremos
   - Quando a distribuição é muito assimétrica
   - Para algoritmos sensíveis a outliers (regressão linear, SVM)

# ==================================================
# 8.1 COMPARANDO O IMPACTO EM DUAS VARIÁVEIS
# ==================================================

Quando comparamos **duas variáveis simultaneamente**, o impacto dos outliers fica ainda mais evidente.
Os outliers podem causar um **"estrangulamento"** dos dados - a maioria dos pontos fica comprimida
em uma pequena região do espaço, dificultando a distinção entre eles. Isso é especialmente problemático
para algoritmos baseados em distância como KNN ou clustering.

In [ ]:
# @title
# Selecionando duas variáveis com presença de outliers
var1 = 'MedInc'  # Renda média
var2 = 'AveOccup'  # Ocupação média (tem muitos outliers)

# Preparando os dados para comparação
data_2vars = X[[var1, var2]].copy()

# Adicionando outliers sintéticos extremos para demonstrar o efeito de estrangulamento
np.random.seed(42)
n_outliers = 50

# Criando outliers extremos
outliers_var1 = np.random.uniform(15, 20, n_outliers)  # Valores muito altos para MedInc
outliers_var2 = np.random.uniform(50, 100, n_outliers)  # Valores extremamente altos para AveOccup

# Criando DataFrame com outliers
outliers_df = pd.DataFrame({
    var1: outliers_var1,
    var2: outliers_var2
})

# Combinando dados originais com outliers sintéticos
data_2vars_with_outliers = pd.concat([data_2vars, outliers_df], ignore_index=True)

# Aplicando os três métodos de escalonamento nos dados com outliers
data_minmax = MinMaxScaler().fit_transform(data_2vars_with_outliers)
data_standard = StandardScaler().fit_transform(data_2vars_with_outliers)
data_robust = RobustScaler().fit_transform(data_2vars_with_outliers)

In [ ]:
# @title
# Visualizando o efeito de "estrangulamento" com outliers sintéticos
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Função para identificar outliers (usando IQR nos dados originais)
def identify_outliers(data):
    Q1 = np.percentile(data[:len(X)], 25, axis=0)  # Usa apenas dados originais para calcular IQR
    Q3 = np.percentile(data[:len(X)], 75, axis=0)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers_mask = np.any((data < lower) | (data > upper), axis=1)
    return outliers_mask

# Identificando outliers (incluindo os sintéticos)
outliers_mask = identify_outliers(data_2vars_with_outliers.values)

# Plot 1: Dados Originais
axes[0].scatter(data_2vars_with_outliers.iloc[~outliers_mask, 0],
                data_2vars_with_outliers.iloc[~outliers_mask, 1],
                alpha=0.6, s=20, label='Dados normais', color='blue')
axes[0].scatter(data_2vars_with_outliers.iloc[outliers_mask, 0],
                data_2vars_with_outliers.iloc[outliers_mask, 1],
                color='red', s=30, alpha=0.8, label='Outliers')
axes[0].set_title('ORIGINAL\nDados em escala real', fontweight='bold')
axes[0].set_xlabel(var1)
axes[0].set_ylabel(var2)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Min-Max Scaling - Mostra estrangulamento severo
axes[1].scatter(data_minmax[~outliers_mask, 0], data_minmax[~outliers_mask, 1],
                alpha=0.6, s=20, label='Dados normais', color='blue')
axes[1].scatter(data_minmax[outliers_mask, 0], data_minmax[outliers_mask, 1],
                color='red', s=30, alpha=0.8, label='Outliers')
axes[1].set_title('MIN-MAX SCALING\n Estrangulamento severo', fontweight='bold', color='darkred')
axes[1].set_xlabel(f'{var1} (normalizado)')
axes[1].set_ylabel(f'{var2} (normalizado)')
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Adicionar zoom para mostrar onde os dados normais estão comprimidos
axes[1].axhspan(0, 0.1, alpha=0.2, color='yellow')
axes[1].axvspan(0, 0.3, alpha=0.2, color='yellow')
axes[1].annotate('Dados normais\ncomprimidos aqui!', xy=(0.05, 0.05),
                xytext=(0.5, 0.3), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5))

# Plot 3: Standardization - Estrangulamento moderado
axes[2].scatter(data_standard[~outliers_mask, 0], data_standard[~outliers_mask, 1],
                alpha=0.6, s=20, label='Dados normais', color='blue')
axes[2].scatter(data_standard[outliers_mask, 0], data_standard[outliers_mask, 1],
                color='red', s=30, alpha=0.8, label='Outliers')
axes[2].set_title('STANDARDIZATION\n Estrangulamento moderado', fontweight='bold', color='darkorange')
axes[2].set_xlabel(f'{var1} (padronizado)')
axes[2].set_ylabel(f'{var2} (padronizado)')
axes[2].set_xlim(-2, 4)
axes[2].set_ylim(-2, 15)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

# Plot 4: Robust Scaling - Melhor preservação
axes[3].scatter(data_robust[~outliers_mask, 0], data_robust[~outliers_mask, 1],
                alpha=0.6, s=20, label='Dados normais', color='blue')
axes[3].scatter(data_robust[outliers_mask, 0], data_robust[outliers_mask, 1],
                color='red', s=30, alpha=0.8, label='Outliers')
axes[3].set_title('ROBUST SCALING\n Melhor preservação do espaço', fontweight='bold', color='darkgreen')
axes[3].set_xlabel(f'{var1} (robust)')
axes[3].set_ylabel(f'{var2} (robust)')
axes[3].set_xlim(-4, 8)
axes[3].set_ylim(-2, 30)
axes[3].legend(fontsize=8)
axes[3].grid(True, alpha=0.3)

plt.suptitle('Comparação do Efeito de "Estrangulamento" com Outliers Extremos',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

# Estatísticas para quantificar o estrangulamento
print("📊 ANÁLISE QUANTITATIVA DO ESTRANGULAMENTO:\n")
print("Porcentagem do intervalo ocupada pelos dados normais (sem outliers):")
print("-" * 60)

# Min-Max
normal_minmax = data_minmax[~outliers_mask]
print(f"MIN-MAX SCALING:")
print(f"  Var1 ({var1}): {normal_minmax[:, 0].max() - normal_minmax[:, 0].min():.1%} do intervalo [0,1]")
print(f"  Var2 ({var2}): {normal_minmax[:, 1].max() - normal_minmax[:, 1].min():.1%} do intervalo [0,1]")
print(f"  → Dados normais ocupam menos de 10% do espaço disponível!\n")

# Standard
normal_std = data_standard[~outliers_mask]
std_range_1 = data_standard[:, 0].max() - data_standard[:, 0].min()
std_range_2 = data_standard[:, 1].max() - data_standard[:, 1].min()
print(f"STANDARDIZATION:")
print(f"  Var1 ({var1}): {(normal_std[:, 0].max() - normal_std[:, 0].min())/std_range_1:.1%} do intervalo total")
print(f"  Var2 ({var2}): {(normal_std[:, 1].max() - normal_std[:, 1].min())/std_range_2:.1%} do intervalo total")
print(f"  → Dados normais ainda sofrem compressão significativa\n")

# Robust
normal_robust = data_robust[~outliers_mask]
robust_range_1 = data_robust[:, 0].max() - data_robust[:, 0].min()
robust_range_2 = data_robust[:, 1].max() - data_robust[:, 1].min()
print(f"ROBUST SCALING:")
print(f"  Var1 ({var1}): {(normal_robust[:, 0].max() - normal_robust[:, 0].min())/robust_range_1:.1%} do intervalo total")
print(f"  Var2 ({var2}): {(normal_robust[:, 1].max() - normal_robust[:, 1].min())/robust_range_2:.1%} do intervalo total")
print(f"  → Dados normais mantêm melhor distribuição no espaço")

### 📊 Comparação das Interpretações

| Técnica de Escalonamento | Intervalo de Referência | Como interpretar os **38.8%** e **0.3%** |
|--------------------------|-------------------------|------------------------------------------|
| **Min-Max Scaling**      | [0, 1] definido pelos valores mínimos e máximos | Dados normais ocupam apenas uma pequena faixa do intervalo total. Outliers “esticam” os extremos, comprimindo os valores centrais. |
| **Standardization (Z-score)** | Intervalo em desvios padrão em torno da média | Dados normais ainda aparecem comprimidos se houver outliers, pois eles aumentam a variância total. |
| **Robust Scaling**       | Intervalo entre \(Q1\) e \(Q3\) (IQR) | Percentual igual, mas agora calculado em um espaço menos sensível a outliers. A distribuição dos dados normais fica mais representativa. |

---

✅ **Resumo:** os percentuais (38.8% e 0.3%) são os mesmos, mas o **intervalo de referência muda**.  
- Min-Max → [min, max]  
- Standard → desvios da média  
- Robust → quartis (IQR)  

Isso altera **como interpretamos a compressão dos dados normais** frente aos outliers.


---

## 🎯 Micro-atividade Guiada: Escolhendo o Scaler Ideal (10 min)

**Objetivo**: Aprender a analisar características dos dados para escolher o método de escalonamento apropriado.

**Cenário**: Você faz parte da equipe de ciência de dados analisando o mercado imobiliário da Califórnia. Precisa preparar os dados para diferentes análises e deve escolher o método de escalonamento mais apropriado para cada variável.

Vamos trabalhar juntos, passo a passo! 🚀

In [ ]:
# @title
# PASSO 1: Selecionando as variáveis para análise
# Vamos trabalhar com três variáveis com características diferentes

variaveis_analise = ['MedInc', 'AveOccup', 'HouseAge']

print("📊 Variáveis selecionadas para análise:")
print("-" * 50)
for var in variaveis_analise:
    print(f"• {var}")

print("\n📝 Lembre-se:")
print("• MedInc: Renda mediana do distrito (em dezenas de milhares de dólares)")
print("• AveOccup: Número médio de moradores por casa")
print("• HouseAge: Idade mediana das casas no distrito (em anos)")

In [ ]:
# @title
# PASSO 2: Criando visualizações para cada variável
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for i, var in enumerate(variaveis_analise):
    # Histograma
    axes[i, 0].hist(X[var], bins=50, edgecolor='black', alpha=0.7)
    axes[i, 0].set_title(f'{var} - Distribuição', fontweight='bold')
    axes[i, 0].set_xlabel('Valor')
    axes[i, 0].set_ylabel('Frequência')
    axes[i, 0].grid(True, alpha=0.3)

    # Boxplot
    box = axes[i, 1].boxplot(X[var], vert=False, patch_artist=True,
                             boxprops=dict(facecolor='lightblue'))
    axes[i, 1].set_title(f'{var} - Boxplot (detectar outliers)', fontweight='bold')
    axes[i, 1].set_xlabel('Valor')
    axes[i, 1].grid(True, alpha=0.3)

plt.suptitle('Análise Visual das Variáveis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔍 O que observar:")
print("• MedInc: Distribuição relativamente normal, muitos outliers não tão extremos à direita")
print("• AveOccup: Outliers extremos! Distribuição muito assimétrica")
print("• HouseAge: Distribuição quase uniforme, sem outliers significativos")

In [ ]:
# @title
# PASSO 3: Calculando estatísticas importantes
from scipy import stats

# Criando DataFrame para armazenar resultados
analise_stats = pd.DataFrame(index=variaveis_analise)

for var in variaveis_analise:
    dados = X[var]

    # Estatísticas básicas
    analise_stats.loc[var, 'Média'] = dados.mean()
    analise_stats.loc[var, 'Mediana'] = dados.median()
    analise_stats.loc[var, 'Desvio Padrão'] = dados.std()

    # Assimetria (Skewness)
    analise_stats.loc[var, 'Assimetria'] = stats.skew(dados)

    # Coeficiente de Variação
    analise_stats.loc[var, 'Coef. Variação'] = dados.std() / dados.mean()

    # Contagem de outliers usando IQR
    Q1 = dados.quantile(0.25)
    Q3 = dados.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((dados < Q1 - 1.5*IQR) | (dados > Q3 + 1.5*IQR)).sum()
    analise_stats.loc[var, 'Qtd Outliers'] = outliers
    analise_stats.loc[var, '% Outliers'] = (outliers / len(dados)) * 100

# Formatando para melhor visualização
print("📊 ANÁLISE ESTATÍSTICA DAS VARIÁVEIS")
print("=" * 80)
print(analise_stats.round(2).to_string())

print("\n📋 INTERPRETAÇÃO:")
print("-" * 80)
print("• MedInc: Assimetria moderada (1.65), alguns outliers (3.30%)")
print("• AveOccup: ALTA assimetria (20.0+), alguns outliers (3.44%)")
print("• HouseAge: Baixa assimetria (0.06), sem outliers (0.0%)")

### PASSO 4: Aplicando e Comparando os Scalers

Agora vamos aplicar os três métodos de escalonamento e observar como cada um trata nossas variáveis:

In [ ]:
# @title
# PASSO 4: Aplicando os três métodos de escalonamento
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Preparando os dados
X_selected = X[variaveis_analise].copy()

# Aplicando cada scaler
scalers = {
    'MinMax': MinMaxScaler(),
    'Standard': StandardScaler(),
    'Robust': RobustScaler()
}

# Dicionário para armazenar resultados
scaled_data = {}

for nome, scaler in scalers.items():
    scaled_data[nome] = pd.DataFrame(
        scaler.fit_transform(X_selected),
        columns=variaveis_analise
    )

print("✅ Scalers aplicados com sucesso!")
print("\nVamos visualizar o efeito de cada um...")

In [ ]:
# @title
# PASSO 4.1: Comparando visualmente o efeito dos scalers
fig, axes = plt.subplots(3, 4, figsize=(18, 12))

for i, var in enumerate(variaveis_analise):
    # Original
    axes[i, 0].boxplot(X[var], vert=False, patch_artist=True,
                       boxprops=dict(facecolor='lightgray'))
    axes[i, 0].set_title(f'{var}\nOriginal', fontweight='bold')
    axes[i, 0].grid(True, alpha=0.3)

    # MinMax
    axes[i, 1].boxplot(scaled_data['MinMax'][var], vert=False, patch_artist=True,
                       boxprops=dict(facecolor='lightblue'))
    axes[i, 1].set_title(f'{var}\nMinMax [0,1]', fontweight='bold')
    axes[i, 1].grid(True, alpha=0.3)
    axes[i, 1].set_xlim(-0.1, 1.1)

    # Standard
    axes[i, 2].boxplot(scaled_data['Standard'][var], vert=False, patch_artist=True,
                       boxprops=dict(facecolor='lightgreen'))
    axes[i, 2].set_title(f'{var}\nStandard (μ=0, σ=1)', fontweight='bold')
    axes[i, 2].grid(True, alpha=0.3)

    # Robust
    axes[i, 3].boxplot(scaled_data['Robust'][var], vert=False, patch_artist=True,
                       boxprops=dict(facecolor='lightyellow'))
    axes[i, 3].set_title(f'{var}\nRobust (median=0)', fontweight='bold')
    axes[i, 3].grid(True, alpha=0.3)

plt.suptitle('Comparação dos Métodos de Escalonamento', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### PASSO 5: Tomando a Decisão - Qual Scaler Usar?

Com base em nossa análise, vamos criar uma tabela de recomendações:

In [ ]:
# @title
# PASSO 5: Criando tabela de recomendações
recomendacoes = pd.DataFrame({
    'Variável': variaveis_analise,
    'Características': [
        'Distribuição ~normal, poucos outliers',
        'MUITOS outliers, extremamente assimétrica',
        'Distribuição uniforme, sem outliers'
    ],
    'Scaler Recomendado': [
        'StandardScaler',
        'RobustScaler',
        'MinMaxScaler ou StandardScaler'
    ],
    'Justificativa': [
        'Dados próximos da normal, StandardScaler preserva bem a distribuição',
        'RobustScaler usa mediana/IQR, não é afetado pelos outliers extremos',
        'Sem outliers, qualquer método funciona bem. MinMax se quiser [0,1]'
    ]
})

# Exibindo a tabela de forma elegante
print("🎯 TABELA DE RECOMENDAÇÕES FINAIS")
print("=" * 100)
for idx, row in recomendacoes.iterrows():
    print(f"\n📌 {row['Variável']}:")
    print(f"   • Características: {row['Características']}")
    print(f"   • Recomendação: {row['Scaler Recomendado']}")
    print(f"   • Razão: {row['Justificativa']}")
print("=" * 100)

### 🎓 Lições Aprendidas

**Processo de decisão para escolher um scaler:**

1. **Analise a distribuição**: Histograma mostra se é normal, assimétrica ou uniforme
2. **Verifique outliers**: Boxplot e cálculo de IQR revelam outliers
3. **Calcule estatísticas**: Assimetria e coeficiente de variação quantificam observações
4. **Compare resultados**: Visualize como cada scaler trata os dados
5. **Tome decisão informada**: Escolha baseada em evidências, não em "achismo"

**Regras práticas:**
- ✅ **Muitos outliers** → RobustScaler
- ✅ **Distribuição normal** → StandardScaler  
- ✅ **Precisa intervalo [0,1]** → MinMaxScaler
- ✅ **Sem outliers significativos** → Qualquer método funciona

---

---

## Resumo da Parte 1

Nesta primeira parte, aprendemos:

1. **Por que escalonar**: Importância para algoritmos baseados em distância e gradiente
2. **Min-Max Scaling**: Normaliza para intervalo [0,1], sensível a outliers
3. **Standardization**: Padroniza para média 0 e desvio padrão 1
4. **Robust Scaling**: Usa mediana e IQR, mais resistente a outliers

**Próxima parte**: Exploraremos técnicas avançadas de transformação e escalonamento, incluindo Power Transformers, Quantile Transformers e como usar pipelines corretamente.